In [1]:
# -*- coding: utf-8 -*-
import sys
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import json

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import _META_COLS

/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------- CONFIG -----------------
parquet_chunks_path = Path("/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/processed/chunks.parquet")
MODEL_NAME     = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE     = 2048

# Entrada: dataset original con los textos
PARQUET_CHUNKS = parquet_chunks_path      

# Salida: carpeta donde guardamos cada batch procesado
OUTPUT_DIR = Path("/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/processed/embeddings_batches")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

META_COLS = _META_COLS

def clean_meta(obj):
    """
    Convierte cualquier valor raro a algo serializable:
    - Timestamp → ISO string (o None si es NaT)
    - numpy types → Python native
    - otros no serializables → str
    """
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat() if pd.notnull(obj) else None
    if isinstance(obj, pd.Timedelta):
        return str(obj)
    if hasattr(obj, "item"):  # numpy scalar (e.g., np.int64, np.float32)
        return obj.item()
    return obj

In [3]:
df = pd.read_parquet(PARQUET_CHUNKS)
df = df.loc[df["chunk_text"].str.strip().fillna("").astype(bool)].reset_index(drop=True)

df

,message_id,parent_id,conversation_id,depth,order_in_conv,role,model_slug,conversation_ttl,created_at,updated_at,chunk_text,chunk_index
0,bbb21b51-9889-499c-8805-3975f3ec2520,33007a2c-1523-4efe-a42d-5219fd469df4,0930598c-baaf-55c9-8117-af1d96d152e4,3,3,user,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:47.508710,<NA>,Qué es el IDCD PROPUESTO Y CUÁLES SON SUS FUNC...,0
1,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,\n## Documento SOTA: Estado del Arte en Interp...,0
2,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,"nte, descomponiendo su computación en unidades...",1
3,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,r respuestas. (Ref: Todos los papers)\n* **S...,2
4,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,dades de Análisis Fundamentales:**\n\n* **Sp...,3
...,...,...,...,...,...,...,...,...,...,...,...,...
224785,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,"o, necesitamos pasarla sin cambios para ser us...",6
224786,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,text\n Answer the question based only on the ...,7
224787,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,omplejo) y lo convierte en un formato de texto...,8
224788,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,"exto relevante, en este caso, ""harrison worked...",9


Wow, defaut haystack text-embedder it's really slow. We might check performance with cProfile..

In [4]:

# import time, cProfile, pstats, io
# docs_prev = []
# documents = []

# doc_embedder.warm_up()
# text_embedder.warm_up()

# for row in df.sample(256).itertuples(index=False):
#     txt = getattr(row, "chunk_text", None)
#     if not (isinstance(txt, str) and txt.strip()):
#         continue

#     meta = {
#         col: getattr(row, col, None)
#         for col in _META_COLS
#         if pd.notna(getattr(row, col, None))
#     }

#     doc = Document(
#         content=txt,
#         meta=meta
#     )
#     docs_prev.append(doc)



# pr = cProfile.Profile()
# pr.enable()

# _ = doc_embedder.run(docs_prev)  # muestra realista
# pr.disable()
# s = io.StringIO(); ps = pstats.Stats(pr, stream=s).sort_stats("cumtime"); ps.print_stats(20)
# print(s.getvalue())


**Lentitud:** El cuello de botella parece ser el propio sentence-transformers..

```bash
9.769 s en /transformers/models/bert/modeling_bert.py:1003(forward) ➔ el BertModel.forward() (el núcleo del encoder).
```

No hay mucho que podamos hacer. Habrá que dejarlo corriendo una tarde.

In [ ]:
# ---------- cargar modelo -------------
model = SentenceTransformer(MODEL_NAME, device="cpu")

# ---------- Detectar progreso previo ---
# Listamos los archivos de batch ya guardados para saber qué lotes están hechos
existing_batches = list(OUTPUT_DIR.glob("batch_*.parquet"))
done_batches = {int(f.stem.split("_")[1]) for f in existing_batches}

total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"{len(done_batches)} batches ya procesados, {total_batches - len(done_batches)} pendientes.")



# ---------- bucle principal ----------
try:
    for batch_number in tqdm(range(total_batches), desc="Batches"):
        if batch_number in done_batches:
            continue  # Skip si este batch ya está hecho

        start = batch_number * BATCH_SIZE
        sub_idx = range(start, min(start + BATCH_SIZE, len(df)))
        batch_df = df.loc[sub_idx]

        texts = batch_df["chunk_text"].tolist()
        metas = batch_df[META_COLS].to_dict(orient="records")

        # --- Inferencia ---
        embs = model.encode(
            texts,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        # --- Guardar ---
        out = pd.DataFrame({
            "row_id": sub_idx,
            "content": texts,
            "meta": [json.dumps({k: clean_meta(v) for k, v in m.items()}) for m in metas],
            "embedding": [emb.tolist() for emb in embs],
        })

        out_file = OUTPUT_DIR / f"batch_{batch_number:04d}.parquet"
        out.to_parquet(out_file, index=False)
        print(f"✅ Batch {batch_number:04d} guardado: {out_file}")

except KeyboardInterrupt:
    print("\n⏸️ Interrupción manual: progreso salvado, puedes reanudar luego.")


In [ ]:
# index_embeddings_to_milvus.py  ✅ listo‑para‑correr
import os, sys, json, numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from haystack import Document
from src.utils import mivuls_doc_store

# ---------- configuración ----------
BATCHES_DIR   = ROOT / "data/processed/embeddings_batches"
COLLECTION    = "openai_chats"
EMB_DIM       = 384
WRITE_BATCH   = 512            # nº docs por inserción
NUKE          = False          # True ⇒ borra colección antes de indexar

ACCEPTED = (bool, int, float, str)    # tipos que Milvus soporta en metadatos
MARK_EXT = ".ok"                      # extensión de marcador “batch procesado”


if NUKE:
    print("🧨  Borrando colección…")
    mivuls_doc_store.delete_documents()

print(f"✅ Conectado a Milvus · colección: {COLLECTION}")

# ---------- helpers ----------
def clean_meta_safe(d: dict) -> dict:
    """Castea valores a tipos Milvus‑friendly; descarta los no soportados."""
    safe = {}
    for k, v in d.items():
        # None →  se omite
        if v is None:
            continue
        # Pandas & NumPy scalars
        if hasattr(v, "item"):
            v = v.item()
        # Timestamps a ISO
        if "Timestamp" in str(type(v)) or "datetime" in str(type(v)):
            v = str(v)
        # Finalmente: comprobar tipo
        if isinstance(v, ACCEPTED):
            safe[k] = v
    return safe

def row_to_doc(row) -> Document:
    meta = json.loads(row["meta"])
    meta = clean_meta_safe(meta)
    emb  = np.asarray(row["embedding"], dtype=np.float32)
    return Document(content=row["content"], embedding=emb, meta=meta)


In [ ]:
# ---------- indexación con reanudación ----------
files = sorted(BATCHES_DIR.glob("batch_*.parquet"))
print(f"📑  {len(files)} lotes de embeddings encontrados en {BATCHES_DIR}")

total_new = 0
for f in tqdm(files, desc="Indexando"):
    marker = f.with_suffix(f"{f.suffix}{MARK_EXT}")          # p.ej. batch_0001.parquet.ok
    if marker.exists():
        continue                                            # lote ya procesado

    df   = pd.read_parquet(f)
    docs = [row_to_doc(r) for _, r in df.iterrows()]

    # inserción por sub‑batches para evitar overflows
    for i in range(0, len(docs), WRITE_BATCH):
        mivuls_doc_store.write_documents(docs[i : i + WRITE_BATCH])

    total_new += len(docs)
    marker.touch()                                          # crea marca de “hecho”

print(f"\n🎉  Indexados {total_new:,} documentos nuevos.")
print(f"🔢  Total en Milvus ahora: {mivuls_doc_store.count_documents():,}")
